# Intermediate NN

In [1]:
#########################     LIBRARIES     ##########################
from keras.models import Model
from keras.layers import Dense, Input
#from keras.layers.merge import concatenate
from tensorflow.keras.layers import concatenate     # PROBLEMA SEMBRA RISOLTO COSI !!!
import keras.backend as K
from keras.regularizers import l2
from hyperopt import STATUS_OK, tpe, Trials, hp, fmin
from hyperopt.pyll.stochastic import sample
from sklearn.model_selection import KFold
import numpy as np
from matplotlib import pyplot as plt
from math import pi
from keras.optimizers import Adam,Nadam,Adamax
from time import perf_counter
import pandas as pd
import pickle
import os
import keras
import tensorflow as tf

from module_utils import * 
sys.path.append('../utils')
from Structure import *

from pathlib import Path

# path to the current notebook
current_file_path = Path().resolve()
# path to the current folder
load_context_functions(current_file_path.parent.name)


True

In [2]:
# reproducibility
seed = 42
np.random.seed(seed)
keras.utils.set_random_seed(seed)
tf.random.set_seed(seed)

## Data Preparation

In [3]:
# import of the discretization and diffusion values
Discretizations= np.loadtxt(
    "../DATA_reaction/Discretizations.txt"
).astype(
    int
)[::-1]

diffusion= np.loadtxt(
    "../DATA_reaction/diffusion.txt"
).astype(
    int
)

In [4]:
########################     PREPARATION      ##########################
file_path_HF = "../DATA_reaction/reaction_diffusion_HF.mat"
(reaction_HF_test, U_HF_test) = import_data(file_path_HF)

U_HF_test = U_HF_test[:, -1, 44,44]

reaction_HF_test = normalization(reaction_HF_test)
U_HF_test=normalization(U_HF_test)

U_HF_test_original=U_HF_test
reaction_HF_test_original=np.c_[reaction_HF_test, np.abs(np.sin(5*np.pi*reaction_HF_test[:, 0]-5*np.pi/6))]

In [5]:
noise_stddev1=[0.01,0.005,0.02]
noise_stddev2=[0.005,0.003,0.01]

(U_HF_test,reaction_HF_test)=add_noise(noise_stddev1,noise_stddev2,reaction_HF_test,U_HF_test)
reaction_HF_test=np.c_[reaction_HF_test, np.abs(np.sin(5*np.pi*reaction_HF_test[:, 0]-5*np.pi/6))]

In [6]:
n_HF = np.array([100])
Nlf = np.array([200])

batch_size=300
Nepo=10000#10000


r2_df = pd.DataFrame(columns=['Discretization','diffusion','R2'])  # dataframe which stores HF R^2
mse_df = pd.DataFrame(columns=['Discretization','diffusion','MSE'])  # dataframe which stores HF R^2
r2_HF_df = pd.DataFrame(columns=['Discretization','diffusion','R2'])  # dataframe which stores HF R^2
r2_LF_df = pd.DataFrame(columns=['Discretization','diffusion','R2'])  # dataframe which stores LF R^2
mse_HF_df = pd.DataFrame(columns=['Discretization','diffusion','MSE'])  # dataframe which stores HF MSE
mse_LF_df = pd.DataFrame(columns=['Discretization','diffusion','MSE'])  # dataframe which stores LF MSE



U_HF_list = []
U_LF_list = []

In [7]:
# Name of the folder
folder_name = "Intermediate_step_models"
folder_path = os.path.join(os.getcwd(), folder_name)

# create the folder
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"Folder '{folder_name}' created.")
else:
    print(f"Folder '{folder_name}' already exists.")
               


for m in range(len(Discretizations)):
    
    test_mse_HF_list = []
    test_mse_LF_list = []
    test_mse_list = []

    r2_HF_list = []
    r2_LF_list = []
    r2_list = []

    for d in range(len(diffusion)):
        for nhf in n_HF:
            for nlf in Nlf:
                
                print(
                f"********************  # Nb. nodes = {Discretizations[m]}  ********************"
                )
                
                print(
                f"********************  # Diff: = {diffusion[d]}  ********************"
                )
                
                print(
                f"********************  # NHF: = {nhf}  ********************"
                )
                print(
                f"********************  # NLF: = {nlf}  ********************"
                )
                
                ### IMPORT LF DATASETs

                file_path_LF = "../DATA_reaction/reaction_diffusion_LF_"+str(Discretizations[m])+"_d"+str(diffusion[d])+".mat"
                (reaction_LF_test, U_LF_test) = import_data(file_path_LF)
                
                U_LF_test = U_LF_test[:,- 1,int(4*(Discretizations[m]-1)/9),int(4*(Discretizations[m]-1)/9)]     
                reaction_LF_test = normalization(reaction_LF_test)
                U_LF_test = normalization(U_LF_test)
                
                reaction_LF_test_original=np.c_[reaction_LF_test, np.abs(np.sin(5*np.pi*reaction_LF_test[:, 0]-5*np.pi/6))]
                U_LF_test_original=U_LF_test 
                
                noise_stddev1=[0.01,0.005,0.02]
                noise_stddev2=[0.005,0.003,0.01]

                (U_LF_test,reaction_LF_test)=add_noise(noise_stddev1,noise_stddev2,reaction_LF_test,U_LF_test)
                
                # randomization
                permutation = np.random.permutation(len(reaction_HF_test))         
                reaction_HF = reaction_HF_test[permutation,:][0:nhf,:]
                U_HF = U_HF_test[permutation][0:nhf]

                permutation = np.random.permutation(len(reaction_LF_test))
                reaction_LF = reaction_LF_test[permutation][0:nlf]
                U_train_LF = U_LF_test[permutation][0:nlf]

                reaction_LF=np.c_[reaction_LF, np.abs(np.sin(5*np.pi*reaction_LF[:, 0]-5*np.pi/6))]
                reaction_LF_test=np.c_[reaction_LF_test, np.abs(np.sin(5*np.pi*reaction_LF_test[:, 0]-5*np.pi/6))]
                
                
                print(
                f"********************  # Nb. nodes = {Discretizations[m]}  ********************"
                )
                
                print(
                f"********************  # Diff: = {diffusion[d]}  ********************"
                )
                
                print(
                f"********************  # NHF: = {nhf}  ********************"
                )
                print(
                f"********************  # NLF: = {nlf}  ********************"
                )

                            
                MAX_EVAL = 30
                #best paramters obtained by HPO:
                #best_params = {'alpha': 0.031884991755260814, 'epochs': 2.0, 'kernel_init': 'uniform', 'l2weight': 0.002651788904350721, 'lr': 0.00042698780348019073, 'nodes': 114.0, 'opt': 'Adamax'}
                best_params = {
                               'alpha': 0.0965167240033655, 
                               'epochs': 3.0, 
                               'kernel_init': 'uniform', 
                               'l2weight': 0.00021154214219451555, 
                               'lr': 0.0065338127394483905, 
                               'nodes': 128.0, 
                               'opt': 'Adamax'
                               }

                names='Inter'
                params=best_params
                reaction_norm=[reaction_LF,reaction_HF]
                reaction_norm_test=[reaction_LF_test,reaction_HF_test]
                reaction_norm_test_original=[reaction_LF_test_original,reaction_HF_test_original]

                definition_inter = {
                                    "network_type": "Inter",
                                    "names": names,
                                    "params": best_params,
                                    "data_train": reaction_norm,
                                    "output_train": [U_HF,U_train_LF],
                                    "N": Nepo * int(best_params['epochs']),
                                    "n": batch_size,
                                    "train": True,
                                    "do_HPO": False,
                                    "verbose": False
                                    }


                model= NetworkFactory.build_network(**definition_inter)
            
                U_pred = model.prediction(reaction_norm_test)
                
                U_Pred = np.concatenate((U_pred[0],U_pred[1]),axis=1)
                U_HF_list.append(U_pred[0][:,0])
                U_LF_list.append(U_pred[1][:,0])
                U_test = np.concatenate((U_HF_test,U_LF_test))

                test_mse= np.mean(np.square(U_test- U_Pred[:,0]))
                test_mse_list.append(test_mse)
                print(f"Test MSE: {test_mse:.8f}")

                test_mse_HF = np.mean(np.square(U_HF_test_original- model.prediction(reaction_norm_test_original)[0][:U_HF_test_original.shape[0],0]))
                test_mse_HF_list.append(test_mse_HF)
                print(f"Test MSE HF: {test_mse_HF:.8f}")

                test_mse_LF = np.mean(np.square(U_LF_test_original- model.prediction(reaction_norm_test_original)[1][U_LF_test_original.shape[0]:,0]))
                test_mse_LF_list.append(test_mse_LF)
                print(f"Test MSE LF: {test_mse_LF:.8f}")

                r2 = 1 - np.sum(np.square(U_test- U_Pred[:,0])) / np.sum(np.square(U_test - np.mean(U_test)))
                r2_list.append(r2)
                print(f"R^2: {r2:.4f}")

                r2_HF = 1 - np.sum(np.square(U_HF_test_original- model.prediction(reaction_norm_test_original)[0][:U_HF_test_original.shape[0],0])) / np.sum(np.square(U_HF_test_original - np.mean(U_HF_test_original)))
                r2_HF_list.append(r2_HF)
                print(f"R^2 HF: {r2_HF:.4f}")

                r2_LF = 1 - np.sum(np.square(U_LF_test_original- model.prediction(reaction_norm_test_original)[1][U_LF_test_original.shape[0]:,0])) / np.sum(np.square(U_LF_test_original - np.mean(U_LF_test_original)))
                r2_LF_list.append(r2_LF)
                print(f"R^2 LF: {r2_LF:.4f}")

                print('\n \n')

                # Primo grafico (LF)
                plt.figure()

                # LF model line: blu tratteggiata e di spessore medio
                plt.plot(reaction_LF_test_original[:, 0], U_LF_test_original, color="#1F77B4", linestyle="--", linewidth=2.5, label="LF model")

                # LF training points: blu, stessi colore e dimensioni dei punti
                plt.plot(
                    reaction_LF[:, 0],
                    U_train_LF,
                    "o",
                    markersize=6,
                    color="#1F77B4",
                    alpha=0.8,
                    label="LF training points",
                )

                # Predicted LF model: verde lime, linea spessa
                plt.plot(
                    reaction_LF_test_original[:, 0],
                    model.prediction(reaction_norm_test)[1][U_LF_test_original.shape[0]:,0],
                    color="#2CA02C",
                    linestyle="-",
                    linewidth=3,
                    label="Predicted LF model",
                )

                # Legenda e griglia
                plt.legend(prop={"size": 9}, loc="best", frameon=True, fancybox=False, shadow=False, facecolor="white", edgecolor="black")
                plt.grid(True, which='both', linestyle=':', linewidth=0.5)

                plt.show()

                # Secondo grafico (HF)
                plt.figure()

                # HF model line: rosso solido, spessore medio
                plt.plot(reaction_HF_test_original[:, 0], U_HF_test_original, color="#FF7F0E", linestyle="-", linewidth=2.5, label="HF model")

                # HF training points: rosso, stessi colore e dimensioni dei punti
                plt.plot(
                    reaction_HF[:, 0],
                    U_HF,
                    "o",
                    markersize=6,
                    color="#FF7F0E",
                    alpha=0.8,
                    label="HF training points",
                )

                # LF training points: blu, stessi colore e dimensioni dei punti
                plt.plot(
                    reaction_LF[:, 0],
                    U_train_LF,
                    "o",
                    markersize=6,
                    color="#1F77B4",
                    alpha=0.8,
                    label="LF training points",
                )

                # Predicted HF model: verde lime, linea spessa
                plt.plot(
                    reaction_HF_test_original[:, 0],
                    model.prediction(reaction_norm_test_original)[0][:U_HF_test.shape[0],0],
                    color="#2CA02C",
                    linestyle="-",
                    linewidth=3,
                    label="Predicted HF model",
                )

                # Legenda e griglia
                plt.legend(prop={"size": 9}, loc="best", frameon=True, fancybox=False, shadow=False, facecolor="white", edgecolor="black")
                plt.grid(True, which='both', linestyle=':', linewidth=0.5)

                plt.show()


                # Creazione dei DataFrame iniziali
                r2_HF_df = pd.DataFrame(columns=['Discretization', 'diffusion', 'R2'])  # dataframe which stores HF R^2
                r2_LF_df = pd.DataFrame(columns=['Discretization', 'diffusion', 'R2'])  # dataframe which stores LF R^2
                mse_HF_df = pd.DataFrame(columns=['Discretization', 'diffusion', 'MSE'])  # dataframe which stores HF MSE
                mse_LF_df = pd.DataFrame(columns=['Discretization', 'diffusion', 'MSE'])  # dataframe which stores LF MSE

                # All'interno del for loop

                new = {'Discretization': Discretizations[m], 'diffusion': diffusion[d], 'R2': r2_LF_list[-1]}
                r2_LF_df = pd.concat([r2_LF_df, pd.DataFrame([new])], ignore_index=True)

                new = {'Discretization': Discretizations[m], 'diffusion': diffusion[d], 'R2': r2_HF_list[-1]}
                r2_HF_df = pd.concat([r2_HF_df, pd.DataFrame([new])], ignore_index=True)

                new = {'Discretization': Discretizations[m], 'diffusion': diffusion[d], 'MSE': test_mse_LF_list[-1]}
                mse_LF_df = pd.concat([mse_LF_df, pd.DataFrame([new])], ignore_index=True)

                new = {'Discretization': Discretizations[m], 'diffusion': diffusion[d], 'MSE': test_mse_HF_list[-1]}
                mse_HF_df = pd.concat([mse_HF_df, pd.DataFrame([new])], ignore_index=True)



                # if len(r2_HF_list[:-1])>0 and r2_HF_list[-1]<max(r2_HF_list[:-1]):
                # #     save_model(finalModel,"finalModel2NN.h5")
                
                
                ##  SALVATAGGIO CON NUOVA CLASSE?
                print(r2_HF_list)
                if not r2_HF_list[:-1]:
                    model.save()
                elif r2_HF_list[:-1] and r2_HF_list[-1] > max(r2_HF_list[:-1]) and test_mse_HF_list[-1]<min(test_mse_HF_list[:-1]):
                    model.save()
        
print(r2_HF_df.round(5))
print(mse_HF_df.round(5))

Folder 'Intermediate_step_models' already exists.
********************  # Nb. nodes = 46  ********************
********************  # Diff: = 0  ********************
********************  # NHF: = 100  ********************
********************  # NLF: = 200  ********************
********************  # Nb. nodes = 46  ********************
********************  # Diff: = 0  ********************
********************  # NHF: = 100  ********************
********************  # NLF: = 200  ********************


ValueError: Exception encountered when calling FourierLayer.call().

[1mDimensions must be equal, but are 64 and 128 for '{{node functional_1/fourier_layer_1_2/Mul}} = Mul[T=DT_FLOAT](functional_1/dense_1_2/add_1, functional_1/fourier_layer_1_2/Mul/ReadVariableOp)' with input shapes: [300,64], [128].[0m

Arguments received by FourierLayer.call():
  • x=tf.Tensor(shape=(300, 64), dtype=float32)

In [13]:
len(reaction_norm)


2

In [ ]:

    U_pred = model.prediction(reaction_norm_test)
    
    U_Pred = np.concatenate((U_pred[0],U_pred[1]),axis=1)
    U_HF_list.append(U_pred[0][:,0])
    U_LF_list.append(U_pred[1][:,0])
    U_test = np.concatenate((U_HF_test,U_LF_test))

    test_mse= np.mean(np.square(U_test- U_Pred[:,0]))
    test_mse_list.append(test_mse)
    print(f"Test MSE: {test_mse:.8f}")

    test_mse_HF = np.mean(np.square(U_HF_test_original- model.model_list[0].prediction(reaction_HF_test_original)[0][:,0]))
    test_mse_HF_list.append(test_mse_HF)
    print(f"Test MSE HF: {test_mse_HF:.8f}")

    test_mse_LF = np.mean(np.square(U_LF_test- model.model_list[0].prediction(reaction_LF_test)[1][:,0]))
    test_mse_LF_list.append(test_mse_LF)
    print(f"Test MSE LF: {test_mse_LF:.8f}")

    r2 = 1 - np.sum(np.square(U_test- U_Pred[:,0])) / np.sum(np.square(U_test - np.mean(U_test)))
    r2_list.append(r2)
    print(f"R^2: {r2:.4f}")

    r2_HF = 1 - np.sum(np.square(U_HF_test_original- model.model_list[0].prediction(reaction_HF_test_original)[0][:,0])) / np.sum(np.square(U_HF_test_original - np.mean(U_HF_test_original)))
    r2_HF_list.append(r2_HF)
    print(f"R^2 HF: {r2_HF:.4f}")

    r2_LF = 1 - np.sum(np.square(U_LF_test- model.model_list[0].prediction(reaction_LF_test)[1][:,0])) / np.sum(np.square(U_LF_test - np.mean(U_LF_test)))
    r2_LF_list.append(r2_LF)
    print(f"R^2 LF: {r2_LF:.4f}")

    print('\n \n')

    plt.figure()
    plt.plot(
    reaction_LF_test[:,0], U_LF_test, "y-", linewidth=1.5, label="LF model"
    )
    plt.plot(
    reaction_LF[:,0],
    U_train_LF,
    "yo",
    markersize=5,
    label="LF training points",
    )
    plt.plot(
    reaction_LF_test[:,0],
    model.model_list[0].prediction(reaction_LF_test)[1],
    "g-",
    linewidth=3,
    label="Predicted LF model",
    )
    plt.legend(prop={"size": 8.3})
    plt.show()
    
    plt.figure()
    plt.plot(
    reaction_HF_test[:,0], U_HF_test, "r-", linewidth=1.5, label="HF model"
    )
    plt.plot(
    reaction_HF[:,0],
    U_HF,
    "ro",
    markersize=5,
    label="HF training points",
    )
    
    order=np.argsort(reaction_LF[:,0])
    
    plt.plot(
    reaction_LF[order,0],
    U_train_LF[order],
    "y-",
    markersize=5,
    label="LF training points",
    )
    
    plt.plot(
    reaction_HF_test[:,0],
    model.model_list[0].prediction(reaction_HF_test)[0],
    "g-",
    linewidth=3,
    label="Predicted HF model",
    )
    plt.legend(prop={"size": 8.3})
    plt.show()

    
    new = {'Discretization': Discretizations[m], 'diffusion': diffusion[d], 'R2': r2_LF_list[-1]}
    r2_LF_df = r2_LF_df.append(new, ignore_index=True)      
    new = {'Discretization': Discretizations[m], 'diffusion': diffusion[d], 'R2': r2_HF_list[-1]}
    r2_HF_df = r2_HF_df.append(new, ignore_index=True)  
    new = {'Discretization': Discretizations[m], 'diffusion': diffusion[d], 'MSE': test_mse_LF_list[-1]}
    mse_LF_df = mse_LF_df.append(new, ignore_index=True) 
    new = {'Discretization': Discretizations[m], 'diffusion': diffusion[d], 'MSE': test_mse_HF_list[-1]}
    mse_HF_df = mse_HF_df.append(new, ignore_index=True)

TypeError: list indices must be integers or slices, not tuple

In [ ]:
reaction_norm_test[0]

array([[0.00000000e+00, 5.00000000e-01],
       [2.04081633e-02, 7.47419595e-01],
       [4.08163265e-02, 9.18685725e-01],
       [6.12244898e-02, 9.96348338e-01],
       [8.16326531e-02, 9.72494508e-01],
       [1.02040816e-01, 8.49554665e-01],
       [1.22448980e-01, 6.40054967e-01],
       [1.42857143e-01, 3.65341024e-01],
       [1.63265306e-01, 5.34030307e-02],
       [1.83673469e-01, 2.63976118e-01],
       [2.04081633e-01, 5.54459135e-01],
       [2.24489796e-01, 7.88449138e-01],
       [2.44897959e-01, 9.42105237e-01],
       [2.65306122e-01, 9.99771641e-01],
       [2.85714286e-01, 9.55572806e-01],
       [3.06122449e-01, 8.14012085e-01],
       [3.26530612e-01, 5.89512890e-01],
       [3.46938776e-01, 3.04949106e-01],
       [3.67346939e-01, 1.06854859e-02],
       [3.87755102e-01, 3.25231350e-01],
       [4.08163265e-01, 6.06639877e-01],
       [4.28571429e-01, 8.26238774e-01],
       [4.48979592e-01, 9.61653437e-01],
       [4.69387755e-01, 9.99086667e-01],
       [4.897959

In [ ]:
reaction_norm=np.concatenate((reaction_HF,reaction_LF),axis=0)
print(reaction_norm.shape)
print(U_pred[0].shape)

(300, 2)
(400, 1)


In [ ]:
#########################     SAVE the OUTPUT      ##########################
os.makedirs('Output_new_Lin')

r2_HF_df.to_csv('./Output_new_Lin/r2_HF_lhs.txt', header=True, index=False, sep='\t', mode='a')
mse_HF_df.to_csv('./Output_new_Lin/mse_HF_lhs.txt', header = True, index = False, sep = '\t', mode = 'a')
r2_LF_df.to_csv('./Output_new_Lin/r2_LF_lhs.txt', header=True, index=False, sep='\t', mode='a')
mse_LF_df.to_csv('./Output_new_Lin/mse_LF_lhs.txt', header = True, index = False, sep = '\t', mode = 'a')


with open('./Output_new_Lin/U_HF_list.data', 'wb') as filehandle:
    # store the data as binary data stream
    pickle.dump(U_HF_list, filehandle)

with open('./Output_new_Lin/U_LF_list.data', 'wb') as filehandle:
    pickle.dump(U_LF_list, filehandle)
